# ResNet From Scratch — Colab runner

Runs an experiment from `experiments/*/experiment.yaml` on a Colab GPU and writes the
results back to Google Drive (so a dropped session loses nothing) and, optionally, to the
GitHub repo.

**Before running:** set `Runtime -> Change runtime type -> T4 GPU`, then fill in the two
variables in the next cell.


In [ ]:
# --- experiment settings ---
REPO_URL = "https://github.com/Szin7/resnet-research.git"
BRANCH = "research-completion-2026"  # switch to main after merging the reviewed changes
EXPERIMENT = "experiments/01_depth/experiment.yaml"
DRIVE_DIR = "/content/drive/MyDrive/resnet-research"


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, pathlib
pathlib.Path(DRIVE_DIR, 'data').mkdir(parents=True, exist_ok=True)
pathlib.Path(DRIVE_DIR, 'results').mkdir(parents=True, exist_ok=True)


In [ ]:
# Check out the selected branch. Keep the tracked repository content intact.
import os
if not os.path.isdir("/content/repo/.git"):
    !git clone --branch $BRANCH $REPO_URL /content/repo
%cd /content/repo
!git fetch origin $BRANCH
!git checkout $BRANCH
!git pull --ff-only origin $BRANCH

# CIFAR downloads can persist on Drive; write experimental results there too.
DATA_ROOT = os.path.join(DRIVE_DIR, "data")
RUNS_ROOT = os.path.join(DRIVE_DIR, "results", "runs")
os.makedirs(DATA_ROOT, exist_ok=True)
os.makedirs(RUNS_ROOT, exist_ok=True)


In [ ]:
!pip -q install -e . torchinfo

import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')


In [ ]:
# Prepare the dataset on Drive
!python scripts/prepare_data.py --root $DATA_ROOT


In [ ]:
# Run a planned experiment. The run directory is on Drive and survives session interruptions.
!python scripts/run_experiment.py --experiment $EXPERIMENT --device cuda --num-workers 2 --data-root $DATA_ROOT --runs-dir $RUNS_ROOT


In [ ]:
# Evaluate the finished runs on the held-out test set.
import glob, os
for run in sorted(glob.glob(os.path.join(RUNS_ROOT, "*"))):
    if os.path.isfile(os.path.join(run, "summary.json")):
        !python scripts/evaluate.py --run $run --device cuda --data-root $DATA_ROOT


In [ ]:
# Rebuild and display plots from the run metrics.
!python scripts/make_figures.py --experiment $EXPERIMENT --runs-dir $RUNS_ROOT
from IPython.display import Image, display
import glob
for png in sorted(glob.glob(os.path.join(os.path.dirname(EXPERIMENT), "figures", "*.png"))):
    display(Image(png))


## Keeping the results

Run data, model checkpoints and evaluation summaries are stored in Google Drive under `resnet-research/results/runs/`. Review the completed metrics before copying selected small tables and figures into the Git repository. Never commit access tokens, raw datasets or large checkpoints.
